In [9]:
import pandas as pd
import sys
import spacy
import re
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

In [5]:
train_df = pd.read_csv("sarcasm.train.tsv.gz", sep="\t", compression="gzip", keep_default_na = False)
bal_df = pd.read_csv("sarcasm.test-bal.tsv.gz", sep="\t", compression="gzip", keep_default_na = False)


In [7]:
print(f"{train_df.label.mean()} are sarcastic in training")
print(f"{bal_df.label.mean()} are sarcastic in training")

0.5 are sarcastic in training
0.5 are sarcastic in training


In [8]:
y_train = list(train_df.label)
y_bal = list(bal_df.label)

In [11]:
#only considers words occuring at least 100 times in the data
#filters stop words.
vectorizer = TfidfVectorizer(min_df = 100, stop_words="english")
X_train_nostop = vectorizer.fit_transform(train_df["text"])
X_bal_nostop = vectorizer.transform(bal_df["text"])

In [14]:
#train classifier
clf_nostop = LogisticRegression()
clf_nostop.fit(X_train_nostop, y_train)

#predict
pred_bal_nostop = clf_nostop.predict(X_bal_nostop)

#evaluate
f1_bal = f1_score(y_bal, pred_bal_nostop)
print(f1_bal)

0.6039107055564608


In [15]:
#Don't use stop words
vectorizer_full = TfidfVectorizer(min_df = 100)
X_train_full = vectorizer_full.fit_transform(train_df.text) 
X_bal_full = vectorizer_full.transform(bal_df.text)

In [16]:
#train classifier
clf_full = LogisticRegression()
clf_full.fit(X_train_full, y_train)

#predict
pred_bal_full = clf_full.predict(X_bal_full)

#evaluate
f1_bal_full = f1_score(y_bal, pred_bal_full)
print(f1_bal_full)


0.631518109427177


In [19]:
# examine what features are informative for the classifier.
coefs = clf_full.coef_
features = vectorizer_full.get_feature_names_out()
coefs, features

(array([[ 0.14573574,  1.01092662,  0.06244902, ...,  0.69878512,
         -0.20085312,  0.09252695]], shape=(1, 2292)),
 array(['000', '10', '100', ..., 'yup', 'zero', 'zone'],
       shape=(2292,), dtype=object))

In [21]:
coef_df = pd.DataFrame(
    {"feat": features,
    "coef": coefs.flatten()}
).set_index("feat")
coef_df

,coef
feat,
000,0.145736
10,1.010927
100,0.062449
1000,0.151280
11,-0.082966
...,...
yourself,0.438967
youtube,-0.243682
yup,0.698785


In [28]:
#top 10 most imformative text
ordered_coef_df = coef_df.sort_values(by= "coef", ascending=False)
top_10 = ordered_coef_df.head(10)

#bottom 10 most imformative text
bottom_10 = ordered_coef_df.tail(10)

In [29]:
top_10, bottom_10

(               coef
 feat               
 clearly    4.153063
 obviously  3.981854
 totally    3.766748
 dare       3.732048
 yeah       3.467570
 shocked    3.203641
 sure       3.162233
 amirite    3.012620
 surely     2.869552
 gee        2.868015,
                coef
 feat               
 tbh       -1.875604
 sad       -1.888266
 pretty    -1.904628
 wish      -1.920297
 reminds   -2.043230
 confused  -2.046739
 sometimes -2.081076
 likely    -2.103620
 lmao      -2.212160
 honestly  -2.560420)

In [35]:
#How many informative features are stopwords?
stop_words = ENGLISH_STOP_WORDS

stop_df = coef_df[coef_df.index.isin(stop_words)]

#take especially high or low to mean above 1.5 or below -1.5
stop_df = stop_df[(stop_df["coef"].abs() > 1.5)]
stop_df.sort_values(by = "coef", ascending = True)

,coef
feat,
sometimes,-2.081076
either,-1.683600
all,1.505362
such,1.577014
but,1.753238
therefore,1.950955
must,2.447220
because,2.583121


In [40]:
pred_probs = clf_full.predict_proba(X_bal_full)[:,1]
pred_df = pd.DataFrame({'pred': pred_bal_full,
                       'label': y_bal,
                       'pred_pr': pred_probs})
pred_df["correct"] = pred_df["pred"] == pred_df["label"]
pred_df

,pred,label,pred_pr,correct
0,1,1,0.891909,True
1,0,0,0.452851,True
2,1,1,0.577010,True
3,0,0,0.499897,True
4,1,1,0.860533,True
...,...,...,...,...
64661,0,0,0.385731,True
64662,0,0,0.223850,True
64663,0,1,0.399502,False
64664,1,1,0.635774,True


In [43]:
def get_cutoff_f1(true, pr, cutoff):
    #returns f1 score for all predictions above a given cutoff
    over_cutoff = (pr >= cutoff).astype(int)
    return f1_score(true, over_cutoff)

#proportion of items the models predicts is "sarcastic", if it makes that prediction 
# when its probability estimate is greater than or equal to a given cutoff.
def get_pr_pos_preds(pr, cutoff):
    over_cutoff = pr[pr >= cutoff]
 
    return len(over_cutoff) / len(y_bal)

In [44]:
for cutoff in [0.5, 0.55]:
    print('cutoff:',cutoff)
    print('f1:', get_cutoff_f1(pred_df.label, pred_df.pred_pr, cutoff))
    print('pr predicted sarcastic:', 
          get_pr_pos_preds(pred_df.pred_pr, cutoff))
    print()

cutoff: 0.5
f1: 0.631518109427177
pr predicted sarcastic: 0.46322642501469086

cutoff: 0.55
f1: 0.5882269503546099
pr predicted sarcastic: 0.3721739399375251



In [51]:
#catch false positives and false negatives
pred_df["text"] = bal_df["text"].values
false_pos = pred_df[(pred_df["pred"] == 1) & (pred_df["label"] == 0)]
 
false_neg = pred_df[(pred_df["pred"] == 0) & (pred_df["label"] == 1)]
 
#get top
false_pos = false_pos.sort_values(by = "pred_pr", ascending = False).head(10)
false_neg = false_neg.sort_values(by = "pred_pr", ascending = True).head(10)

In [52]:
false_pos

,pred,label,pred_pr,correct,text
49140,1,0,0.988443,False,Yeah because this is totally sustainable.
58191,1,0,0.981689,False,"Wow, totally racist scum, and the shooter too"
36201,1,0,0.977415,False,"Oh, yeah, I totally wouldn't have understood t..."
7319,1,0,0.973889,False,"Yeah, sure you could."
19200,1,0,0.972188,False,Obviously: MOCHA USON BLOG.
48211,1,0,0.972188,False,Obviously.
56935,1,0,0.969731,False,Yeah because he can!
13763,1,0,0.968738,False,Yeah because they totally don't allow jew jokes.
62396,1,0,0.968010,False,Clearly workplace violence.
5204,1,0,0.967748,False,He's clearly joking


In [53]:
false_neg

,pred,label,pred_pr,correct,text
6739,0,1,0.045681,False,ayy lmao fuck razer
27203,0,1,0.052921,False,Sometimes during Liturgy I wish I was doped up.
30772,0,1,0.065441,False,"I think its Rizzo and Bryant, honestly."
51588,0,1,0.066612,False,HahahhHahahahah lmao
30745,0,1,0.066647,False,Reminds me of Seth Rollins last night.
7837,0,1,0.071609,False,You realize a shitpost is actually a shitpost ...
37192,0,1,0.077664,False,I'm confused.
50057,0,1,0.079023,False,Holy shit this creeps me the fuck out
3627,0,1,0.084641,False,I fucking wish.
39776,0,1,0.087287,False,"Fucking hyped, fuck vandoorne that little shit!"


In [55]:
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(vectorizer_full, clf_full)

def get_single_prediction(text, pipe=pipe):
    return pipe.predict([text])[0]

In [ ]:
#predict custom texts
print(get_single_prediction('At least he said thank you'))

1
